# Pump-and-Dump Security Detection

This notebook is the primary analysis and pipeline runner for the stock price manipulation detection project.

**What we're trying to do:** Identify securities that will be used in a pump-and-dump scheme at least 5 days before the fraud occurs — so the fraud team can place targeted trading blocks in time.

**Story arc:**
1. Acquire confirmed P&D cases from SEC enforcement actions
2. Filter to US/Canadian equities and clean the dataset
3. Understand what these securities look like before the fraud (price + volume EDA)
4. Determine how far back we need to look (observation window analysis)
5. Understand the class imbalance problem
6. Engineer features and preview the feature matrix
7. Set up for TN acquisition and modelling

**Two constraints that shape everything:**
- Only ~20 confirmed cases in the original production dataset (larger set from SEC here)
- The model must fire ≥5 days before D so fraud ops can act in time

---
### Portfolio data note

All data in this notebook is sourced from public APIs: confirmed P&D cases from **SEC EDGAR** enforcement actions and price/volume history from **yfinance**. The original production system used proprietary brokerage data — customer transaction records, internal account demographics, and trade logs — which cannot be shared publicly.

The analysis is scoped to **US and Canadian equities** as a portfolio simplification. In the original system, no country filter was applied; the model ran across all securities on the brokerage's platform. Here the scope is narrowed because SEC enforcement data is predominantly US-focused and yfinance coverage is strongest for North American exchanges.

---
## 0. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data.sec_scraper import SECPumpDumpScraper
from src.data.downloader import MarketDataDownloader
from src.data.preprocessor import Preprocessor
from src.features.feature_transformer import FeatureTransformer
from src.features.feature_config import CHANGE_WINDOWS, ROLLING_WINDOWS, PRICE_COLS

EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
RAW_DIR      = PROJECT_ROOT / 'data' / 'raw'
PROC_DIR     = PROJECT_ROOT / 'data' / 'processed'
PROC_DIR.mkdir(exist_ok=True)

TP_CSV   = EXTERNAL_DIR / 'tp_tickers.csv'
TP_OHLCV = RAW_DIR / 'tp_ohlcv.parquet'
TP_META  = RAW_DIR / 'tp_metadata.csv'

LOOKBACK_DAYS = 90
BUFFER_DAYS   = 5

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 4)

print('Setup complete')

---
## 1. Data Acquisition

We source confirmed pump-and-dump cases from **SEC litigation releases** — public enforcement documents that name specific tickers and dates. This gives us real true positives with real fraud dates (D dates), which anchor every observation window downstream.

Cells check for locally cached data first and skip re-downloading if it already exists.

In [ ]:
# --- 1a: Load confirmed P&D cases ---
if TP_CSV.exists():
    print(f'Loading cached TP cases from {TP_CSV}')
    cases_df = pd.read_csv(TP_CSV, parse_dates=['d_date'])
else:
    print('Scraping SEC litigation releases for pump-and-dump cases...')
    scraper = SECPumpDumpScraper(output_dir=EXTERNAL_DIR)
    cases_df = scraper.fetch_confirmed_cases(start_date='2010-01-01', end_date='2023-12-31')
    scraper.save(cases_df, 'tp_tickers.csv')

print(f'\nTotal TP ticker/date pairs : {len(cases_df)}')
print(f'Unique tickers             : {cases_df["ticker"].nunique()}')
print(f'Date range                 : {cases_df["d_date"].min().date()} → {cases_df["d_date"].max().date()}')
cases_df.head(10)

In [ ]:
# --- 1b: Download OHLCV and metadata via yfinance ---
if TP_OHLCV.exists() and TP_META.exists():
    print('Loading cached OHLCV and metadata...')
    ohlcv_raw = pd.read_parquet(TP_OHLCV)
    meta_raw  = pd.read_csv(TP_META, parse_dates=['d_date'])
else:
    print(f'Downloading for {len(cases_df)} tickers  (window D-{LOOKBACK_DAYS} to D-{BUFFER_DAYS})...')
    # download_batch expects a 'case_date' column as the D date
    dl_df = cases_df.rename(columns={'d_date': 'case_date'})
    downloader = MarketDataDownloader(output_dir=RAW_DIR)
    ohlcv_raw, meta_raw = downloader.download_batch(
        dl_df, lookback_days=LOOKBACK_DAYS, buffer_days=BUFFER_DAYS, label=1
    )
    downloader.save_parquet(ohlcv_raw, 'tp_ohlcv.parquet')
    downloader.save_csv(meta_raw, 'tp_metadata.csv')

print(f'\nRaw OHLCV rows   : {len(ohlcv_raw):,}')
print(f'Tickers with data: {ohlcv_raw["ticker"].nunique()}')
ohlcv_raw.head()

---
## 2. Data Scope & Filtering

Before analysis, we apply two filters:

**Country / currency filter** — Restrict to US and Canadian equities. As noted above, the original production system applied no country filter; this is a demo simplification driven by data availability.

**Minimum history filter** — Drop any ticker with fewer than 45 trading days in the observation window. Securities that were halted, thinly traded, or newly listed will have large gaps in their OHLCV history; including them would produce mostly-NaN feature vectors and distort the analysis.

In [ ]:
preprocessor = Preprocessor()  # uses defaults from feature_config.py

print('Filtering dataset...')
ohlcv_df, meta_df = preprocessor.run(ohlcv_raw, meta_raw)
preprocessor.summary(ohlcv_df, meta_df)

print(f'\nRemoved {ohlcv_raw["ticker"].nunique() - ohlcv_df["ticker"].nunique()} tickers total')

In [ ]:
# What got filtered and why?
raw_tickers      = set(ohlcv_raw['ticker'].unique())
filtered_tickers = set(ohlcv_df['ticker'].unique())
removed          = raw_tickers - filtered_tickers

if removed:
    removed_meta = meta_raw[meta_raw['ticker'].isin(removed)][['ticker', 'country', 'currency', 'exchange']]
    print('Removed tickers:')
    display(removed_meta)
else:
    print('No tickers removed — all pass the country and history filters.')

# Country distribution of what remains
if 'country' in meta_df.columns:
    fig, ax = plt.subplots(figsize=(8, 3))
    meta_df['country'].fillna('Unknown').value_counts().plot.barh(ax=ax, color='steelblue')
    ax.set_title('Filtered TP Securities by Country')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()

---
## 3. Dataset Overview

Now that the dataset is clean, let's understand what we're working with: when did these cases occur, which markets are represented, and how complete is the price history?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Cases by year
year_counts = cases_df[
    cases_df['ticker'].isin(ohlcv_df['ticker'].unique())
]['d_date'].dt.year.value_counts().sort_index()
axes[0].bar(year_counts.index, year_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Confirmed P&D Cases by Year (filtered dataset)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Cases')

# Trading days per ticker
days_per_ticker = ohlcv_df.groupby('ticker')['date'].count()
axes[1].hist(days_per_ticker, bins=20, color='coral', edgecolor='white')
axes[1].axvline(LOOKBACK_DAYS, color='black', linestyle='--', label=f'Target: {LOOKBACK_DAYS} days')
axes[1].set_title('Trading Days Available Per TP Ticker')
axes[1].set_xlabel('Trading Days in Window')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Tickers in filtered dataset: {ohlcv_df["ticker"].nunique()}')
print(f'Median trading days:         {days_per_ticker.median():.0f}')

In [ ]:
# Exchange and sector breakdown
if 'exchange' in meta_df.columns and 'sector' in meta_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    meta_df['exchange'].fillna('Unknown').value_counts().head(10).plot.barh(ax=axes[0], color='steelblue')
    axes[0].set_title('TP Securities by Exchange')
    axes[0].set_xlabel('Count')
    meta_df['sector'].fillna('Unknown').value_counts().head(10).plot.barh(ax=axes[1], color='coral')
    axes[1].set_title('TP Securities by Sector')
    axes[1].set_xlabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('Exchange/sector metadata not available.')

---
## 4. Price Behaviour EDA

The central question: **do pump-and-dump securities show a detectable price signature in the days and weeks before the fraud event?**

We align all TP securities to their D date (D=0) and inspect normalised price trajectories. If there's a signal, we expect to see price inflation building in the window — ideally visible before D-5, the model's cutoff.

In [ ]:
def add_days_to_d(df):
    df = df.copy()
    df['date']   = pd.to_datetime(df['date'])
    df['d_date'] = pd.to_datetime(df['d_date'])
    df['days_to_d'] = (df['date'] - df['d_date']).dt.days
    return df

def normalise_to_window_start(df, col):
    df = df.copy()
    first = df.groupby('ticker')[col].transform('first')
    df[f'{col}_norm'] = df[col] / first
    return df

ohlcv_df = add_days_to_d(ohlcv_df)
ohlcv_df = normalise_to_window_start(ohlcv_df, 'close')
print('Days-to-D range:', ohlcv_df['days_to_d'].min(), '→', ohlcv_df['days_to_d'].max())

In [ ]:
# All trajectories + median overlay
fig, ax = plt.subplots(figsize=(14, 5))
for _, grp in ohlcv_df.groupby('ticker'):
    ax.plot(grp['days_to_d'], grp['close_norm'], alpha=0.15, linewidth=0.8, color='steelblue')
med = ohlcv_df.groupby('days_to_d')['close_norm'].median()
ax.plot(med.index, med.values, color='crimson', linewidth=2.5, label='Median across TPs')
ax.axvline(-BUFFER_DAYS, color='black', linestyle='--', linewidth=1.2, label=f'D-{BUFFER_DAYS} (model cutoff)')
ax.axvline(0, color='gray', linestyle=':', linewidth=1, label='D (fraud date)')
ax.set_xlabel('Days Relative to Fraud Date (D)')
ax.set_ylabel('Normalised Close Price (window start = 1.0)')
ax.set_title('Price Trajectories of Confirmed P&D Securities')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Total price change over the window per ticker
window_returns = (
    ohlcv_df.groupby('ticker')
    .apply(lambda g: g.sort_values('days_to_d')['close_norm'].iloc[-1] - 1)
    * 100
).rename('return_pct')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(window_returns, bins=25, color='steelblue', edgecolor='white')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(window_returns.median(), color='crimson', linewidth=2,
           label=f'Median: {window_returns.median():.1f}%')
ax.set_title('Total Price Change Over Observation Window (TP Securities)')
ax.set_xlabel('Price Change from Window Start to D-5 (%)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Median return : {window_returns.median():.1f}%')
print(f'Mean return   : {window_returns.mean():.1f}%')
print(f'Rising into D : {(window_returns > 0).mean():.0%} of TPs')

**Observation:** Price trajectories vary widely — no clean directional trend exists across all TPs. Some securities inflate steadily before D; others show almost no price movement until very close to the fraud event. This is why raw price levels are weak features: they capture noise as much as signal.

This motivates **statistical summaries** of price behaviour: rate-of-change, rolling volatility, and relative position — rather than levels or simple trends.

---
## 5. Volume Behaviour EDA

Volume is often a stronger early signal than price. The pump phase requires sustained buy-side pressure to move the price; the dump creates a net sell spike. Does volume show a more consistent pattern leading up to D?

In [ ]:
mean_vol = ohlcv_df.groupby('ticker')['volume'].transform('mean')
ohlcv_df['volume_norm'] = ohlcv_df['volume'] / mean_vol

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for _, grp in ohlcv_df.groupby('ticker'):
    axes[0].plot(grp['days_to_d'], grp['close_norm'],  alpha=0.12, linewidth=0.8, color='steelblue')
    axes[1].plot(grp['days_to_d'], grp['volume_norm'], alpha=0.12, linewidth=0.8, color='coral')

price_med = ohlcv_df.groupby('days_to_d')['close_norm'].median()
axes[0].plot(price_med.index, price_med.values, color='crimson', linewidth=2.5, label='Median price (norm)')
axes[0].axvline(-BUFFER_DAYS, color='black', linestyle='--')
axes[0].set_ylabel('Normalised Close')
axes[0].set_title('Price and Volume Behaviour of TP Securities Before Fraud Date')
axes[0].legend()

vol_med = ohlcv_df.groupby('days_to_d')['volume_norm'].median()
axes[1].plot(vol_med.index, vol_med.values, color='darkorange', linewidth=2.5, label='Median volume (norm)')
axes[1].axvline(-BUFFER_DAYS, color='black', linestyle='--', label=f'D-{BUFFER_DAYS} cutoff')
axes[1].axhline(1.0, color='gray', linestyle=':', linewidth=1)
axes[1].set_ylabel('Normalised Volume (/ ticker mean)')
axes[1].set_xlabel('Days Relative to Fraud Date (D)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 10-day rolling volume — sustained build-up?
ohlcv_df = ohlcv_df.sort_values(['ticker', 'days_to_d'])
ohlcv_df['volume_roll10'] = (
    ohlcv_df.groupby('ticker')['volume_norm']
    .transform(lambda x: x.rolling(10, min_periods=3).mean())
)

fig, ax = plt.subplots(figsize=(14, 4))
roll_med = ohlcv_df.groupby('days_to_d')['volume_roll10'].median()
roll_q25 = ohlcv_df.groupby('days_to_d')['volume_roll10'].quantile(0.25)
roll_q75 = ohlcv_df.groupby('days_to_d')['volume_roll10'].quantile(0.75)
ax.fill_between(roll_med.index, roll_q25, roll_q75, alpha=0.2, color='coral', label='IQR')
ax.plot(roll_med.index, roll_med.values, color='darkorange', linewidth=2.5, label='Median 10-day rolling vol')
ax.axvline(-BUFFER_DAYS, color='black', linestyle='--', linewidth=1.2, label=f'D-{BUFFER_DAYS} cutoff')
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('Days Relative to D')
ax.set_ylabel('10-Day Rolling Volume (/ ticker mean)')
ax.set_title('Sustained Volume Build-Up Before Fraud Date')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Observation Window Length Analysis

The 90-day window was a business starting point. The key question: **at what point before D does a detectable signal first appear?** If signal only emerges in the final 30 days, a longer window adds noise. If behaviour starts diverging at 90+ days, we may be under-capturing it.

We compare signal consistency across window lengths. A full TP vs. TN comparison will be re-run once TN data is available.

In [ ]:
# 10-day bucket averages — where does signal visually diverge from baseline?
ohlcv_df['window_bucket'] = pd.cut(
    ohlcv_df['days_to_d'],
    bins=range(-LOOKBACK_DAYS - 1, 1, 10)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
price_by_b = ohlcv_df.groupby('window_bucket', observed=True)['close_norm'].median()
vol_by_b   = ohlcv_df.groupby('window_bucket', observed=True)['volume_norm'].median()
labels     = [f'{b.mid:.0f}d' for b in price_by_b.index]

axes[0].bar(range(len(price_by_b)), price_by_b.values, color='steelblue', edgecolor='white')
axes[0].axhline(1.0, color='gray', linestyle=':')
axes[0].set_xticks(range(len(price_by_b)))
axes[0].set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
axes[0].set_title('Median Normalised Price by 10-Day Bucket')
axes[0].set_ylabel('Normalised Close (window start = 1.0)')

axes[1].bar(range(len(vol_by_b)), vol_by_b.values, color='coral', edgecolor='white')
axes[1].axhline(1.0, color='gray', linestyle=':')
axes[1].set_xticks(range(len(vol_by_b)))
axes[1].set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
axes[1].set_title('Median Normalised Volume by 10-Day Bucket')
axes[1].set_ylabel('Normalised Volume (/ ticker mean)')

plt.tight_layout()
plt.show()

In [ ]:
# Volume signal consistency across lookback lengths (TP data only — proxy until TNs available)
lookback_candidates = [30, 45, 60, 90, 120]
rows = []
for lb in lookback_candidates:
    slice_df = ohlcv_df[ohlcv_df['days_to_d'] >= -lb]
    tv = slice_df.groupby('ticker')['volume_norm'].mean()
    rows.append({
        'lookback_days' : lb,
        'n_tickers'     : len(tv),
        'mean_vol'      : tv.mean(),
        'std_vol'       : tv.std(),
        'cv'            : tv.std() / tv.mean() if tv.mean() > 0 else np.nan,
    })

print('Volume signal consistency by lookback window (lower CV = more homogeneous across TPs):')
print(pd.DataFrame(rows).to_string(index=False))
print('\nFull AUC-PR comparison vs. TNs will run once TN OHLCV is available.')

---
## 7. Class Imbalance

Fraudulent securities are a tiny fraction of all tradeable equities. Understanding the scale of imbalance makes clear why the legacy 50% flag rate was operationally untenable and why naive sampling approaches will fail.

In [ ]:
approx_us_equities = 6000  # NASDAQ ~3400 + NYSE ~2400 + AMEX ~300
n_tp = ohlcv_df['ticker'].nunique()

print('=== Class Imbalance ===')
print(f'Approx. US-listed equities   : ~{approx_us_equities:,}')
print(f'Confirmed P&D tickers (TPs)  : {n_tp}')
print(f'Imbalance ratio (TN:TP)      : ~{approx_us_equities // n_tp}:1')
print()
print('Naive baselines:')
print(f'  Predict all negative        → Accuracy {1 - n_tp/approx_us_equities:.1%},  Recall 0%')
legacy_prec = n_tp / (approx_us_equities * 0.5)
print(f'  Legacy rules (~50% flag)    → ~{approx_us_equities//2:,} blocks for {n_tp} real TPs  (precision ≈ {legacy_prec:.2%})')

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.barh(['Equity universe'], [approx_us_equities], color='lightgray', label='True Negatives')
ax.barh(['Equity universe'], [n_tp], color='crimson', label=f'True Positives ({n_tp})')
ax.set_xlabel('Count')
ax.set_title('Class Imbalance: P&D Securities vs. Full Equity Universe')
ax.legend()
plt.tight_layout()
plt.show()

**Why random undersampling isn't enough:** Randomly picking 5 TNs per TP likely gives you a large-cap S&P 500 stock alongside a micro-cap P&D target. The classifier trivially separates them on market cap alone and learns nothing useful. Our centroid-based approach assigns each TN to its nearest TP by behavioural profile, so the classifier must learn genuine distinctions between similar-looking securities.

---
## 8. Feature Engineering

The `FeatureTransformer` converts each ticker's OHLCV time series into a flat feature vector. It computes:

| Group | Features |
|---|---|
| Price changes | N-day % change of close, open, high, low at horizons 1/5/10/15/30/45/90 days |
| Overnight gap | % change: today's open vs. yesterday's close — same horizons |
| Rolling stats | Mean, std, positive-day count of daily % changes over 5/10/15/30/45/90-day windows |
| Intraday | Rolling mean/std of: intraday range, close-minus-open, high-minus-close, close-minus-low |
| Volume | Raw + log-scaled N-day changes; rolling mean, std, sum, positive-day count |
| Metadata | One-hot: country, currency, exchange; binned market cap tier |

**Scaling note:** Features are returned unscaled here. Robust scaling (median/IQR) is applied only inside `CentroidSelector` for computing centroid distances. The Decision Tree receives unscaled features to keep splits interpretable.

In [ ]:
transformer = FeatureTransformer()
feat_names  = transformer.feature_names

print(f'Total time-series features : {len(feat_names)}')
print(f'(+ metadata one-hot columns added when metadata_df is provided)\n')

# Show feature groups and counts
groups = {
    'Price N-day changes (close/open/high/low)'   : [f for f in feat_names if any(f.startswith(p+'_chg') for p in PRICE_COLS)],
    'Gap N-day changes'                           : [f for f in feat_names if f.startswith('gap_chg')],
    'Price rolling stats (close/open/high/low)'   : [f for f in feat_names if any(f.startswith(p+'_roll') for p in PRICE_COLS)],
    'Gap rolling stats'                           : [f for f in feat_names if f.startswith('gap_roll')],
    'Intraday rolling stats'                      : [f for f in feat_names if any(f.startswith(m) for m in ['intraday','close_minus_open','high_minus','close_minus_low'])],
    'Volume changes'                              : [f for f in feat_names if f.startswith('volume_chg') or f.startswith('volume_log')],
    'Volume rolling stats'                        : [f for f in feat_names if f.startswith('volume_roll')],
}
for name, feats in groups.items():
    print(f'  {name:<50s}: {len(feats):>3d} features')

print(f'\nFirst 10 feature names:')
print(feat_names[:10])

In [ ]:
# Run transformer on the TP dataset
print('Computing features for TP securities...')
tp_features = transformer.transform(ohlcv_df, metadata_df=meta_df, label_col='label')

print(f'Feature matrix shape : {tp_features.shape}')
print(f'Tickers              : {tp_features["ticker"].nunique()}')
print(f'NaN rate             : {tp_features[feat_names].isna().mean().mean():.1%} (across all features)')

# Save for downstream modelling
tp_features.to_parquet(PROC_DIR / 'tp_features.parquet', index=False)
print(f'\nSaved to {PROC_DIR / "tp_features.parquet"}')

tp_features.head(3)

In [ ]:
# Distribution of a selection of key features across TP securities
sample_feats = [
    'close_chg_5d',  'close_chg_30d', 'close_chg_90d',
    'volume_chg_5d', 'volume_chg_30d',
    'close_roll30_mean', 'volume_roll30_mean',
    'intraday_range_roll30_mean',
]
available = [f for f in sample_feats if f in tp_features.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for i, feat in enumerate(available[:8]):
    vals = tp_features[feat].dropna()
    axes[i].hist(vals, bins=20, color='steelblue', edgecolor='white')
    axes[i].axvline(vals.median(), color='crimson', linewidth=1.5, label=f'Median: {vals.median():.3f}')
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)

for j in range(len(available), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions Across TP Securities', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of the sample features (TP securities)
corr_feats = [f for f in sample_feats if f in tp_features.columns]
if len(corr_feats) > 2:
    corr = tp_features[corr_feats].corr()
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
                linewidths=0.5, annot_kws={'size': 8})
    ax.set_title('Feature Correlation (TP Securities)')
    plt.tight_layout()
    plt.show()

---
## 9. Next Steps: True Negative Acquisition

Features for TP securities are now computed. To build the training dataset we need TN securities.

**TN acquisition workflow:**
1. **Source candidates** — Download a broad universe of North American-listed tickers (NASDAQ/NYSE listing files are public CSVs)
2. **Assign D dates** — Each TN inherits the D date of its nearest TP centroid (initially matched by market cap + sector)
3. **Download OHLCV** — Same D-90 to D-5 window; same `MarketDataDownloader` 
4. **Apply `FeatureTransformer`** — Compute the same feature set on TN OHLCV
5. **Centroid assignment in feature space** — Reassign each TN to its nearest TP using Euclidean distance on robust-scaled features
6. **Downsample** — Apply the three methods (random, distance-ranked, stratified-by-distance) and grid-search the optimal configuration

**Baseline:** the legacy rule-based system flagged ~50% of securities (precision ≈ {legacy_prec:.2%}). The ML pipeline should substantially improve precision at equal or better recall.

In [ ]:
print('TN acquisition plan:')
print(f'  TPs available              : {n_tp}')
print(f'  TN:TP ratio (baseline)     : 5:1  →  ~{n_tp * 5} TNs to download')
print(f'  Grid search ratios         : 1:1, 3:1, 5:1, 10:1')
print(f'  Distance bin candidates    : Q = 4, 5, 10')
print()
print('Implementation status:')
print('  [x] src/features/feature_transformer.py')
print('  [ ] src/models/centroid_selector.py')
print('  [ ] src/models/classifier.py')
print('  [ ] src/models/isolation_forest.py')
print('  [ ] src/pipeline/training_pipeline.py')
print('  [ ] TN download + centroid assignment')

---
## 10. Summary

| Finding | Implication |
|---|---|
| Price trajectories vary widely across TPs | Raw price levels are weak; use statistical summaries (rolling stats, rate-of-change) |
| Volume may show earlier / more consistent signal | Prioritise volume-based features in engineering |
| ~50% legacy flag rate → precision ≈ <1% | A modest ML model should dramatically outperform the baseline |
| Small TP set | LOO-CV mandatory; no leakage between M1 selection and M2 evaluation |
| Observation window (90 days) not yet validated | Re-run window-length comparison once TN data is available |
| Features computed and saved | `data/processed/tp_features.parquet` ready for TN merge and modelling |